<a href="https://colab.research.google.com/github/sergiojsp/gnn-practicas/blob/main/visualizar_grafo_movies_neo4j.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install neo4j pyvis

**Importación de librerías**

In [ ]:
from neo4j import GraphDatabase   # importa el cliente oficial de Neo4j para conectar a la base de datos y ejecutar consultas
from pyvis.network import Network # importa funciones de Pyvis para crear visualizaciones interactivas de grafos en HTML
from IPython.display import HTML, display  # importa funciones de IPython para mostrar contenido HTML directamente en un Jupyter Notebook

**Conexión a Neo4j AuraDB**

In [ ]:
# Datos Aura DB
uri = "neo4j+s://escribe_el_id_instancia.databases.neo4j.io"
user = "neo4j"
password = "escribe tu contraseña"

driver = GraphDatabase.driver(uri, auth=(user, password))

**Consultar el Grafo completo**

In [ ]:
def obtener_grafo(tx):
    query = """
    MATCH (n)-[r]->(m)
    RETURN n, r, m
    """
    return list(tx.run(query))

with driver.session() as session:
    resultados = session.execute_read(obtener_grafo)

**Procesamiento y visualización del grafo**

In [ ]:
net = Network(height='750px', width='100%', notebook=True, cdn_resources='remote')

nodos_agregados = set()

for record in resultados:
    nodo1 = record["n"]
    nodo2 = record["m"]
    relacion = record["r"]

    if nodo1.element_id not in nodos_agregados:
         label = nodo1.get("name") if "name" in nodo1 else list(nodo1.labels)[0] if nodo1.labels else "Nodo"
         net.add_node(nodo1.element_id, label=label, color='#90D5FF',  title=str(dict(nodo1)))
         nodos_agregados.add(nodo1.element_id)

    if nodo2.element_id not in nodos_agregados:
         label = nodo2.get("title") if "title" in nodo2 else list(nodo2.labels)[0] if nodo2.labels else "Nodo"
         net.add_node(nodo2.element_id, label=label, color='#88E788', title=str(dict(nodo2)))
         nodos_agregados.add(nodo2.element_id)
    net.add_edge(nodo1.element_id, nodo2.element_id, label=relacion.type)

archivo_html = "grafo_notebook.html"
net.show(archivo_html)
display(HTML(archivo_html))
driver.close()
